<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/06_Daten_Transformieren.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔄 Notebook 06: Daten transformieren

**Szenario**: Bereite den Datensatz für ein ML-Modell vor – reshapen, encoden, skalieren.

**Lernziele:**
- melt(), pivot(), pivot_table()
- One-Hot-Encoding und Label-Encoding
- MinMaxScaler und StandardScaler
- Feature Engineering

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(BASE_URL + "customer_churn.csv")
print(f'Datensatz: {df.shape}')
print(df.head(3))

## 1️⃣ Reshape: melt(), pivot(), pivot_table()

In [ ]:
# Beispiel: Umsatzdaten nach Quartal (wide format)
df_wide = pd.DataFrame({
    'produkt': ['A', 'B', 'C'],
    'Q1': [100, 200, 150],
    'Q2': [120, 180, 160],
    'Q3': [130, 220, 140],
    'Q4': [110, 190, 170]
})
print('Wide Format:')
print(df_wide)

# melt(): wide → long
df_long = df_wide.melt(id_vars=['produkt'], var_name='quartal', value_name='umsatz')
print('\nLong Format (nach melt):')
print(df_long.head(8))

# pivot(): long → wide
df_back = df_long.pivot(index='produkt', columns='quartal', values='umsatz')
df_back.columns.name = None
print('\nZurück zu Wide (nach pivot):')
print(df_back)

In [ ]:
# pivot_table: mit Aggregation (wie Excel Pivot)
df_pivot = df.pivot_table(
    index='contract_type',
    columns='internet_service',
    values='monthly_charge',
    aggfunc='mean'
).round(2)
print('Pivot Table: Ø monatliche Kosten nach Vertragstyp und Internetservice:')
print(df_pivot)

## 2️⃣ Encoding kategorischer Variablen

In [ ]:
df_enc = df.copy()

# One-Hot-Encoding (nominale Kategorien)
print('Vor Encoding:', df_enc['contract_type'].unique())

df_enc = pd.get_dummies(df_enc, columns=['contract_type', 'internet_service'],
                        drop_first=True, prefix=['contract','internet'])
print('\nNach One-Hot-Encoding:')
new_cols = [c for c in df_enc.columns if c.startswith('contract') or c.startswith('internet')]
print('Neue Spalten:', new_cols)
print(df_enc[new_cols].head(3))

In [ ]:
# Label-Encoding (ordinale Kategorien)
le = LabelEncoder()
df['tech_support_enc'] = le.fit_transform(df['tech_support'])
print('Label-Encoding tech_support:')
for orig, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {orig} → {enc}')

## 3️⃣ Feature Scaling

In [ ]:
num_cols = ['monthly_charge', 'total_charges', 'tenure_months']
df_num = df[num_cols].dropna()

print('Vorher:')
print(df_num.describe().round(2))

# MinMaxScaler [0, 1]
scaler_mm = MinMaxScaler()
df_normalized = pd.DataFrame(scaler_mm.fit_transform(df_num), columns=[c+'_norm' for c in num_cols])
print('\nNach MinMaxScaler [0,1]:')
print(df_normalized.describe().round(3))

# StandardScaler (Z-Score)
scaler_std = StandardScaler()
df_standardized = pd.DataFrame(scaler_std.fit_transform(df_num), columns=[c+'_std' for c in num_cols])
print('\nNach StandardScaler (mean=0, std=1):')
print(df_standardized.describe().round(3))

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, [df_num[num_cols[0]], df_normalized[num_cols[0]+'_norm'], df_standardized[num_cols[0]+'_std']],
                           ['Original', 'MinMax Normalized', 'Standardized']):
    ax.hist(col, bins=30, color='#0389CF', alpha=0.7, edgecolor='white')
    ax.set_title(f'monthly_charge – {title}')
    ax.axvline(col.mean(), color='#E8792F', linestyle='--', label=f'mean={col.mean():.2f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4️⃣ Feature Engineering

In [ ]:
df_feat = df.copy()

# Neue Features ableiten
df_feat['total_charges'] = pd.to_numeric(df_feat['total_charges'], errors='coerce')

# Kosten pro Monat Laufzeit
df_feat['charge_per_tenure'] = (df_feat['total_charges'] / df_feat['tenure_months'].replace(0,1)).round(2)

# Altersgruppen
df_feat['age_group'] = pd.cut(df_feat['age'],
    bins=[0, 30, 45, 60, 100],
    labels=['Jung (≤30)', 'Mittel (31-45)', 'Senior (46-60)', 'Alt (>60)'])

# Langzeitkunde-Flag
df_feat['is_longterm'] = (df_feat['tenure_months'] >= 24).astype(int)

# Hohe Kosten
df_feat['high_charges'] = (df_feat['monthly_charge'] > df_feat['monthly_charge'].quantile(0.75)).astype(int)

print('Neue Features:')
print(df_feat[['tenure_months','charge_per_tenure','age_group','is_longterm','high_charges']].head(8))

## ✅ Challenges

1. Erstelle eine Pivot-Tabelle: Churn-Rate nach `contract_type` und `internet_service`
2. One-Hot-encode den `age_group` Feature den du gerade erstellt hast
3. Normalisiere `monthly_charge` und `tenure_months` mit MinMaxScaler

💡 Tipp: `pd.pivot_table(df, values='churn', index='contract_type', aggfunc='mean')`

In [ ]:
# Deine Lösung:



---
**Weiter mit:** `07_Daten_Zusammenfuehren.ipynb`